<a href="https://colab.research.google.com/github/sila85/production_ml_demo/blob/main/cora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q spektral tensorflow scikit-learn

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from spektral.datasets import Citation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

tf.keras.utils.set_random_seed(42)
np.random.seed(42)

In [ ]:
import scipy.sparse as sp

# Compatibility fix for older Spektral code
if not hasattr(sp.csr_matrix, "A"):
    sp.csr_matrix.A = property(lambda self: self.toarray())

if not hasattr(sp.csc_matrix, "A"):
    sp.csc_matrix.A = property(lambda self: self.toarray())

print("SciPy–Spektral compatibility patch applied.")

In [ ]:
import numpy as np
from spektral.datasets import Citation

dataset = Citation(
    name="cora",
    random_split=False,
    normalize_x=True
)

graph = dataset[0]

X = np.asarray(graph.x, dtype=np.float32)
A = graph.a.toarray().astype(np.float32)
y = np.asarray(graph.y)

if y.ndim == 2:
    y = np.argmax(y, axis=1)

print("Features:", X.shape)
print("Adjacency:", A.shape)
print("Labels:", y.shape)
print("Training nodes:", dataset.mask_tr.sum())
print("Validation nodes:", dataset.mask_va.sum())
print("Test nodes:", dataset.mask_te.sum())

In [ ]:
train_mask = dataset.mask_tr
val_mask = dataset.mask_va
test_mask = dataset.mask_te

num_features = X.shape[1]
num_classes = len(np.unique(y))

In [ ]:
print("Features:", X.shape)
print("Adjacency:", A.shape)
print("Classes:", num_classes if "num_classes" in globals()
      else len(np.unique(y)))
print("Training nodes:", train_mask.sum())
print("Validation nodes:", val_mask.sum())
print("Test nodes:", test_mask.sum())

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

baseline = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_history = baseline.fit(
    X[train_mask],
    y[train_mask],
    validation_data=(X[val_mask], y[val_mask]),
    epochs=200,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        )
    ],
    verbose=1
)

In [ ]:
import tensorflow as tf

class GraphConvolution(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation_fn = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_shape[0][-1], self.units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.bias = self.add_weight(
            name="bias",
            shape=(self.units,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):
        features, adjacency = inputs

        support = tf.matmul(features, self.kernel)
        output = tf.matmul(adjacency, support) + self.bias

        if self.activation_fn is not None:
            output = self.activation_fn(output)

        return output


class GCN(tf.keras.Model):
    def __init__(self, num_classes):
        super().__init__()

        self.gc1 = GraphConvolution(16, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.gc2 = GraphConvolution(num_classes)

    def call(self, inputs, training=False):
        features, adjacency = inputs

        hidden = self.gc1([features, adjacency])
        hidden = self.dropout(hidden, training=training)
        output = self.gc2([hidden, adjacency])

        return output


print("GCN model defined successfully.")

In [ ]:
print("X:", X.shape, X.dtype)
print("A_norm:", A_norm.shape, A_norm.dtype)
print("y:", y.shape, y.dtype)
print("train_mask:", train_mask.shape, train_mask.dtype)
print("val_mask:", val_mask.shape, val_mask.dtype)
print("Classes:", np.unique(y))
print("Number of classes:", num_classes)

In [ ]:
logits = gcn([X, A_norm], training=False)

print("GCN output shape:", logits.shape)
print("Expected shape:", (len(y), num_classes))

In [ ]:
# Ensure consistent data types
X = np.asarray(X, dtype=np.float32)
A_norm = np.asarray(A_norm, dtype=np.float32)
y = np.asarray(y, dtype=np.int32).reshape(-1)

train_mask = np.asarray(train_mask, dtype=bool)
val_mask = np.asarray(val_mask, dtype=bool)
test_mask = np.asarray(test_mask, dtype=bool)

# Verify dimensions
print("X:", X.shape)
print("A_norm:", A_norm.shape)
print("y:", y.shape)

# Reset the model
tf.keras.backend.clear_session()
gcn = GCN(num_classes)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

# Convert data to TensorFlow tensors
X_tf = tf.convert_to_tensor(X)
A_tf = tf.convert_to_tensor(A_norm)
y_tf = tf.convert_to_tensor(y)

train_idx = np.where(train_mask)[0]
val_idx = np.where(val_mask)[0]

@tf.function
def train_step():
    with tf.GradientTape() as tape:
        logits = gcn([X_tf, A_tf], training=True)
        train_logits = tf.gather(logits, train_idx)
        train_labels = tf.gather(y_tf, train_idx)
        loss = loss_fn(train_labels, train_logits)

    gradients = tape.gradient(loss, gcn.trainable_variables)
    optimizer.apply_gradients(
        zip(gradients, gcn.trainable_variables)
    )
    return loss

@tf.function
def validation_step():
    logits = gcn([X_tf, A_tf], training=False)
    val_logits = tf.gather(logits, val_idx)
    val_labels = tf.gather(y_tf, val_idx)
    return loss_fn(val_labels, val_logits)

best_val_loss = float("inf")
best_weights = None
patience = 20
wait = 0

train_losses = []
val_losses = []

for epoch in range(200):
    train_loss = train_step()
    val_loss = validation_step()

    train_losses.append(float(train_loss.numpy()))
    val_losses.append(float(val_loss.numpy()))

    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]
        best_weights = gcn.get_weights()
        wait = 0
    else:
        wait += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Train loss: {train_losses[-1]:.4f} | "
            f"Validation loss: {val_losses[-1]:.4f}"
        )

    if wait >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

if best_weights is not None:
    gcn.set_weights(best_weights)

print("GCN training complete.")

In [ ]:
gcn_logits = gcn([X_tf, A_tf], training=False).numpy()
gcn_preds = np.argmax(gcn_logits, axis=1)

print("GCN test accuracy:",
      accuracy_score(y[test_mask], gcn_preds[test_mask]))

print(classification_report(
    y[test_mask],
    gcn_preds[test_mask],
    zero_division=0
))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Baseline predictions: features only
baseline_probs = baseline.predict(X, verbose=0)
baseline_preds = np.argmax(baseline_probs, axis=1)

# GCN predictions: features + graph structure
gcn_logits = gcn(
    [X_tf, A_tf], training=False
).numpy()
gcn_preds = np.argmax(gcn_logits, axis=1)

# Evaluate only the held-out test nodes
y_test = y[test_mask]

def calculate_metrics(name, predictions):
    pred_test = predictions[test_mask]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred_test),
        "Precision (Macro)": precision_score(
            y_test, pred_test, average="macro", zero_division=0
        ),
        "Recall (Macro)": recall_score(
            y_test, pred_test, average="macro", zero_division=0
        ),
        "F1-score (Macro)": f1_score(
            y_test, pred_test, average="macro", zero_division=0
        )
    }

results = pd.DataFrame([
    calculate_metrics("Baseline MLP", baseline_preds),
    calculate_metrics("GCN", gcn_preds)
])

print(results.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

results.set_index("Model").plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1),
    rot=0
)

plt.title("Baseline MLP vs GCN on Cora")
plt.ylabel("Test Performance")
plt.xlabel("Model")
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
baseline_acc = results.loc[
    results["Model"] == "Baseline MLP", "Accuracy"
].iloc[0]

gcn_acc = results.loc[
    results["Model"] == "GCN", "Accuracy"
].iloc[0]

difference = gcn_acc - baseline_acc

print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"GCN accuracy:      {gcn_acc:.4f}")
print(f"Accuracy change:   {difference:+.4f}")
print(f"Change in percentage points: {difference * 100:+.2f}")

In [ ]:
def build_baseline(num_features, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(num_features,)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


baseline = build_baseline(num_features, num_classes)

baseline_history = baseline.fit(
    X[train_mask],
    y[train_mask],
    validation_data=(X[val_mask], y[val_mask]),
    epochs=200,
    verbose=0,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        )
    ]
)

print("Baseline training complete.")

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

baseline = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

baseline.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

baseline_history = baseline.fit(
    X[train_mask],
    y[train_mask],
    validation_data=(X[val_mask], y[val_mask]),
    epochs=200,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=20,
            restore_best_weights=True
        )
    ],
    verbose=1
)

In [ ]:
import tensorflow as tf

class GraphConvolution(tf.keras.layers.Layer):
    def __init__(self, units, activation=None):
        super().__init__()
        self.units = units
        self.activation_fn = tf.keras.activations.get(activation)

    def build(self, input_shape):
        self.kernel = self.add_weight(
            name="kernel",
            shape=(input_shape[0][-1], self.units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.bias = self.add_weight(
            name="bias",
            shape=(self.units,),
            initializer="zeros",
            trainable=True
        )

    def call(self, inputs):
        features, adjacency = inputs

        support = tf.matmul(features, self.kernel)
        output = tf.matmul(adjacency, support) + self.bias

        if self.activation_fn is not None:
            output = self.activation_fn(output)

        return output


class GCN(tf.keras.Model):
    def __init__(self, num_classes):
        super().__init__()

        self.gc1 = GraphConvolution(16, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.5)
        self.gc2 = GraphConvolution(num_classes)

    def call(self, inputs, training=False):
        features, adjacency = inputs

        hidden = self.gc1([features, adjacency])
        hidden = self.dropout(hidden, training=training)
        output = self.gc2([hidden, adjacency])

        return output


print("GCN model defined successfully.")

In [ ]:
# Ensure consistent data types
X = np.asarray(X, dtype=np.float32)
A_norm = np.asarray(A_norm, dtype=np.float32)
y = np.asarray(y, dtype=np.int32).reshape(-1)

train_mask = np.asarray(train_mask, dtype=bool)
val_mask = np.asarray(val_mask, dtype=bool)
test_mask = np.asarray(test_mask, dtype=bool)

# Verify dimensions
print("X:", X.shape)
print("A_norm:", A_norm.shape)
print("y:", y.shape)

# Reset the model
tf.keras.backend.clear_session()
gcn = GCN(num_classes)

optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

# Convert data to TensorFlow tensors
X_tf = tf.convert_to_tensor(X)
A_tf = tf.convert_to_tensor(A_norm)
y_tf = tf.convert_to_tensor(y)

train_idx = np.where(train_mask)[0]
val_idx = np.where(val_mask)[0]

@tf.function
def train_step():
    with tf.GradientTape() as tape:
        logits = gcn([X_tf, A_tf], training=True)
        train_logits = tf.gather(logits, train_idx)
        train_labels = tf.gather(y_tf, train_idx)
        loss = loss_fn(train_labels, train_logits)

    gradients = tape.gradient(loss, gcn.trainable_variables)
    optimizer.apply_gradients(
        zip(gradients, gcn.trainable_variables)
    )
    return loss

@tf.function
def validation_step():
    logits = gcn([X_tf, A_tf], training=False)
    val_logits = tf.gather(logits, val_idx)
    val_labels = tf.gather(y_tf, val_idx)
    return loss_fn(val_labels, val_logits)

best_val_loss = float("inf")
best_weights = None
patience = 20
wait = 0

train_losses = []
val_losses = []

for epoch in range(200):
    train_loss = train_step()
    val_loss = validation_step()

    train_losses.append(float(train_loss.numpy()))
    val_losses.append(float(val_loss.numpy()))

    if val_losses[-1] < best_val_loss:
        best_val_loss = val_losses[-1]
        best_weights = gcn.get_weights()
        wait = 0
    else:
        wait += 1

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:03d} | "
            f"Train loss: {train_losses[-1]:.4f} | "
            f"Validation loss: {val_losses[-1]:.4f}"
        )

    if wait >= patience:
        print(f"Early stopping at epoch {epoch + 1}")
        break

if best_weights is not None:
    gcn.set_weights(best_weights)

print("GCN training complete.")

In [ ]:
gcn_logits = gcn([X_tf, A_tf], training=False).numpy()
gcn_preds = np.argmax(gcn_logits, axis=1)

print("GCN test accuracy:",
      accuracy_score(y[test_mask], gcn_preds[test_mask]))

print(classification_report(
    y[test_mask],
    gcn_preds[test_mask],
    zero_division=0
))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Baseline predictions: features only
baseline_probs = baseline.predict(X, verbose=0)
baseline_preds = np.argmax(baseline_probs, axis=1)

# GCN predictions: features + graph structure
gcn_logits = gcn(
    [X_tf, A_tf], training=False
).numpy()
gcn_preds = np.argmax(gcn_logits, axis=1)

# Evaluate only the held-out test nodes
y_test = y[test_mask]

def calculate_metrics(name, predictions):
    pred_test = predictions[test_mask]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred_test),
        "Precision (Macro)": precision_score(
            y_test, pred_test, average="macro", zero_division=0
        ),
        "Recall (Macro)": recall_score(
            y_test, pred_test, average="macro", zero_division=0
        ),
        "F1-score (Macro)": f1_score(
            y_test, pred_test, average="macro", zero_division=0
        )
    }

results = pd.DataFrame([
    calculate_metrics("Baseline MLP", baseline_preds),
    calculate_metrics("GCN", gcn_preds)
])

print(results.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

results.set_index("Model").plot(
    kind="bar",
    figsize=(9, 5),
    ylim=(0, 1),
    rot=0
)

plt.title("Baseline MLP vs GCN on Cora")
plt.ylabel("Test Performance")
plt.xlabel("Model")
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
baseline_acc = results.loc[
    results["Model"] == "Baseline MLP", "Accuracy"
].iloc[0]

gcn_acc = results.loc[
    results["Model"] == "GCN", "Accuracy"
].iloc[0]

difference = gcn_acc - baseline_acc

print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"GCN accuracy:      {gcn_acc:.4f}")
print(f"Accuracy change:   {difference:+.4f}")
print(f"Change in percentage points: {difference * 100:+.2f}")